# MoodSwing — Data Science Pipeline

**Goal:** Build a *Mood-to-Town* recommendation engine using synthetic data.

| Section | Content |
|---|---|
| 1 | Synthetic data generation |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Feature engineering |
| 4 | Model training — RF vs XGBoost |
| 5 | Evaluation & export |
| 6 | Town wordclouds |

## 1 · Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data',             exist_ok=True)
os.makedirs('models',           exist_ok=True)
os.makedirs('assets/wordclouds',exist_ok=True)

sns.set_theme(style='whitegrid', palette='pastel')
plt.rcParams['figure.dpi'] = 110
print('Setup complete ✓')

## 2 · Data Generation

Generates `data/synthetic_moods.csv` with realistic per-user behaviour.
Each user has a personality profile controlling how often they feel sad / happy / calm.
Timestamps, notes, and town names match the format of real user data.

- To **regenerate fresh data**, set `FORCE_REGENERATE = True`
- To **keep existing data**, leave it as `False`

In [ ]:
import os
from datetime import datetime, timedelta

# ── Toggle this to regenerate fresh data ──────────────────────────────────
FORCE_REGENERATE = False

# ── Configuration ──────────────────────────────────────────────────────────
N_USERS    = 50
DAYS       = 14
START_DATE = datetime(2026, 2, 19)

MOOD_ID  = {'sad': 0, 'happy': 1, 'calm': 2}
TOWN_MAP = {'sad': 'Mist Haven', 'happy': 'Spark City', 'calm': 'Bloom Fields'}

NOTES = {
    'happy': [
        'Played with my dog', 'Watched a movie', 'Read a book',
        'Cooked a nice meal', 'Lunch with friends', 'Went for a walk',
        'Had a great workout', 'Family time', 'Enjoyed a sunny afternoon',
        'Caught up with an old friend', 'Finished a project I am proud of',
        'Had a really good laugh today', 'Treated myself to something nice',
        'Woke up feeling great', 'Good news arrived today',
    ],
    'sad': [
        'Couldn t get out of bed today', 'Feeling really low',
        'Miss having someone to talk to', 'Everything feels heavy',
        'Cried and didn t even know why', 'Stayed home all day',
        'Just need some quiet time', 'Hard day, feeling drained',
        'Feeling invisible', 'Not in the mood for anything',
        'Called a friend but no answer', 'Looked through old photos',
        'Felt really alone today', 'Too tired to explain how I feel',
        'Sitting in the dark listening to music',
    ],
    'calm': [
        'Just drifting today', 'Taking things slow', 'Went for a quiet walk',
        'Did some journaling', 'Had a slow morning with tea',
        'Not sure what I want right now', 'Feeling a bit lost but okay',
        'Sat by the window and thought', 'Took a long bath',
        'Nothing planned, just being', 'Feeling uncertain about everything',
        'Quiet evening at home', 'Watched the clouds for a while',
        'Tidied my room, felt a bit clearer', 'Listened to lo-fi and did nothing',
    ],
}

NOTE_RATE = {'happy': 1.0, 'sad': 0.65, 'calm': 0.65}

PERSONALITY_TYPES = [
    {'name': 'mostly_happy',   'probs': [0.10, 0.70, 0.20]},
    {'name': 'mostly_sad',     'probs': [0.65, 0.10, 0.25]},
    {'name': 'mostly_calm',    'probs': [0.15, 0.20, 0.65]},
    {'name': 'mixed',          'probs': [0.30, 0.40, 0.30]},
    {'name': 'sad_recovering', 'probs': [0.45, 0.30, 0.25]},
]

PEAK_HOURS = {
    'sad':   (21, 2),
    'happy': (9, 18),
    'calm':  (10, 15),
}

def sample_hour(mood):
    start, end = PEAK_HOURS[mood]
    if np.random.rand() < 0.70:
        if end < start:
            pool = list(range(start, 24)) + list(range(0, end + 1))
            return int(np.random.choice(pool))
        return np.random.randint(start, end + 1)
    return np.random.randint(0, 24)

def pick_note(mood):
    if np.random.rand() < NOTE_RATE[mood]:
        return np.random.choice(NOTES[mood])
    return None

# ── Generate or load ───────────────────────────────────────────────────────
if FORCE_REGENERATE or not os.path.exists('data/synthetic_moods.csv'):
    np.random.seed(42)
    moods = ['sad', 'happy', 'calm']
    rows  = []

    for i in range(1, N_USERS + 1):
        user_id     = f'U{i:03d}'
        personality = np.random.choice(PERSONALITY_TYPES)
        probs       = personality['probs']

        for day_offset in range(DAYS):
            date = START_DATE + timedelta(days=day_offset)
            mood = np.random.choice(moods, p=probs)
            hour = sample_hour(mood)
            ts   = date.replace(
                hour=hour,
                minute=np.random.randint(0, 60),
                second=np.random.randint(0, 60),
            )
            rows.append({
                'user_id':     user_id,
                'mood_id':     MOOD_ID[mood],
                'mood':        mood,
                'note':        pick_note(mood),
                'hour_of_day': hour,
                'day_of_week': ts.weekday(),
                'target_town': TOWN_MAP[mood],
                'timestamp':   ts.strftime('%Y-%m-%d %H:%M:%S'),
            })

    df = (pd.DataFrame(rows)
          .sample(frac=1, random_state=42)
          .reset_index(drop=True))

    df.to_csv('data/synthetic_moods.csv', index=False)
    print(f'Generated {len(df)} rows  ->  data/synthetic_moods.csv')
else:
    df = pd.read_csv('data/synthetic_moods.csv')
    print(f'Loaded existing data  ->  {len(df)} rows  (set FORCE_REGENERATE=True to regenerate)')

print()
print('Town distribution:')
print(df['target_town'].value_counts())
print()
print('Mood distribution:')
print(df['mood'].value_counts())
df.head(10)